[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/27_deep_learning_foundations/03_neural_network_from_scratch.ipynb)

# Bài 03 — Mạng nơ-ron from scratch với NumPy

**🎯 Mục tiêu**
- Thiết kế theo **lớp (layer)**: mỗi lớp có `forward` (lưu cache) và `backward` (nhận $\partial L/\partial\text{out}$, trả $\partial L/\partial\text{in}$, tính gradient tham số).
- Softmax + cross-entropy gộp, ổn định số; gradient $\hat P - Y$.
- Kiểm tra gradient toàn mạng; vòng lặp huấn luyện mini-batch SGD.
- Thử nghiệm: dữ liệu xoắn ốc 3 lớp (độ rộng, độ sâu), chữ số viết tay 8×8; overfitting khi ít dữ liệu.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/27_deep_learning_foundations

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from ml_course.algorithms import MLPScratch
from ml_course.visualization import set_style

set_style()

## 1. Các lớp

| Lớp | forward | backward (nhận `d_out`) |
|---|---|---|
| `Linear` | $Z = XW + \mathbf b$ | $dW = X^\top d_{out}$, $d\mathbf b = \sum d_{out}$, $dX = d_{out}W^\top$ |
| `ReLU` | $\max(0, Z)$ | $d_{out} \odot \mathbb 1[Z > 0]$ |
| `Tanh` | $\tanh Z$ | $d_{out} \odot (1 - \tanh^2 Z)$ |
| `SoftmaxCrossEntropy` | $L = -\frac1n\sum\log\hat p_{i,y_i}$ | $\frac1n(\hat P - Y)$ |

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, rng, init="he"):
        scale = np.sqrt(2.0 / n_in) if init == "he" else np.sqrt(1.0 / n_in)   # khởi tạo — chi tiết ở chương 28
        self.W = rng.normal(0, scale, (n_in, n_out))
        self.b = np.zeros(n_out)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, d_out):
        self.dW = self.X.T @ d_out
        self.db = d_out.sum(axis=0)
        return d_out @ self.W.T

    def params_and_grads(self):
        return [(self.W, self.dW), (self.b, self.db)]


class ReLU:
    def forward(self, Z):
        self.mask = Z > 0
        return Z * self.mask

    def backward(self, d_out):
        return d_out * self.mask

    def params_and_grads(self):
        return []


class Tanh:
    def forward(self, Z):
        self.A = np.tanh(Z)
        return self.A

    def backward(self, d_out):
        return d_out * (1 - self.A**2)

    def params_and_grads(self):
        return []


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        shifted = logits - logits.max(axis=1, keepdims=True)            # trừ max: tránh tràn số exp
        log_probs = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        self.probs, self.y = np.exp(log_probs), y
        return -log_probs[np.arange(len(y)), y].mean()

    def backward(self):
        d = self.probs.copy()
        d[np.arange(len(self.y)), self.y] -= 1                          # P̂ − Y
        return d / len(self.y)

Gradient gọn $\hat P - Y$ là lý do softmax và cross-entropy luôn được **gộp**: tính riêng $\partial L/\partial\hat p = -1/\hat p$ rồi nhân Jacobian softmax vừa tốn kém vừa mất ổn định khi $\hat p \to 0$.

In [ ]:
class Network:
    def __init__(self, sizes, activation="relu", seed=0):
        rng = np.random.default_rng(seed)
        Act = ReLU if activation == "relu" else Tanh
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], rng, init="he" if activation == "relu" else "xavier"))
            if i < len(sizes) - 2:
                self.layers.append(Act())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def loss(self, X, y):
        return self.loss_fn.forward(self.forward(X), y)

    def backward(self):
        d = self.loss_fn.backward()
        for layer in reversed(self.layers):
            d = layer.backward(d)

    def params_and_grads(self):
        return [pg for layer in self.layers for pg in layer.params_and_grads()]

    def predict(self, X):
        return self.forward(X).argmax(axis=1)


def train(net, X, y, X_val=None, y_val=None, lr=0.1, epochs=100, batch_size=64, seed=0, verbose_every=0):
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(epochs):
        perm = rng.permutation(len(X))
        for s in range(0, len(X), batch_size):
            idx = perm[s:s + batch_size]
            net.loss(X[idx], y[idx])
            net.backward()
            for p, g in net.params_and_grads():
                p -= lr * g                                             # cập nhật tại chỗ (in-place)
        rec = {"loss": net.loss(X, y), "train_acc": np.mean(net.predict(X) == y)}
        if X_val is not None:
            rec["val_loss"] = net.loss(X_val, y_val)
            rec["val_acc"] = np.mean(net.predict(X_val) == y_val)
        history.append(rec)
    return history

## 2. Kiểm tra gradient toàn mạng

Trước khi huấn luyện, so gradient backprop với sai phân hữu hạn trên vài tham số ngẫu nhiên:

In [ ]:
rng = np.random.default_rng(1)
net_chk = Network([4, 5, 5, 3], activation="tanh", seed=1)
X_chk, y_chk = rng.normal(size=(8, 4)), rng.integers(0, 3, 8)
net_chk.loss(X_chk, y_chk)
net_chk.backward()
max_rel = 0.0
for p, g in net_chk.params_and_grads():
    for _ in range(3):
        idx = tuple(rng.integers(0, s) for s in p.shape)
        old = p[idx]
        p[idx] = old + 1e-5
        lp = net_chk.loss(X_chk, y_chk)
        p[idx] = old - 1e-5
        lm = net_chk.loss(X_chk, y_chk)
        p[idx] = old
        num = (lp - lm) / 2e-5
        max_rel = max(max_rel, abs(num - g[idx]) / max(abs(num) + abs(g[idx]), 1e-12))
print(f"sai lệch tương đối lớn nhất (18 tham số ngẫu nhiên): {max_rel:.2e}")

Sai lệch ~$10^{-8}$ trở xuống → backprop đúng. (Với ReLU, tránh kiểm tra đúng tại điểm gãy $z = 0$.)

## 3. Dữ liệu xoắn ốc 3 lớp

In [ ]:
def make_spiral(n_per_class=150, n_classes=3, noise=0.2, seed=0):
    r = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        t = np.linspace(0, 1, n_per_class)
        theta = np.linspace(k * 4, (k + 1) * 4, n_per_class) + r.normal(0, noise, n_per_class)
        X.append(np.c_[t * np.sin(theta), t * np.cos(theta)])
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)


X_sp, y_sp = make_spiral()


def plot_boundary(ax, net, X, y, title):
    gx, gy = np.meshgrid(np.linspace(-1.1, 1.1, 200), np.linspace(-1.1, 1.1, 200))
    Z = net.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z, alpha=0.3, cmap="brg")
    ax.scatter(*X.T, c=y, cmap="brg", s=8)
    ax.set_title(title)


configs = [("softmax tuyến tính", [2, 3]), ("1 lớp ẩn × 8", [2, 8, 3]), ("1 lớp ẩn × 64", [2, 64, 3]), ("2 lớp ẩn × 64", [2, 64, 64, 3])]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (name, sizes) in zip(axes, configs):
    net_sp = Network(sizes, seed=0)
    hist = train(net_sp, X_sp, y_sp, lr=0.5, epochs=300, batch_size=64)
    plot_boundary(ax, net_sp, X_sp, y_sp, f"{name}: accuracy train = {hist[-1]['train_acc']:.3f}")
    print(f"{name:20s} loss cuối = {hist[-1]['loss']:.3f} | accuracy train = {hist[-1]['train_acc']:.3f}")
plt.show()

Mô hình tuyến tính không tách được xoắn ốc (~51%). Lớp ẩn nhỏ (8) đã uốn ranh giới khá tốt (~92%); rộng hơn hoặc sâu hơn khớp sát hơn nữa (~96–98%). (Đây là accuracy **train** — để biết mô hình có tổng quát hoá không, cần tập validation như mục sau.)

## 4. Chữ số viết tay 8×8

1 797 ảnh 8×8 (64 pixel) — chuẩn hoá đầu vào, chia train/val/test.

In [ ]:
X_dg, y_dg = load_digits(return_X_y=True)
X_tmp, X_te, y_tmp, y_te = train_test_split(X_dg, y_dg, test_size=0.2, random_state=0, stratify=y_dg)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.2, random_state=0, stratify=y_tmp)
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_va_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)

t0 = time.perf_counter()
net_dg = Network([64, 128, 64, 10], seed=0)
hist_dg = train(net_dg, X_tr_s, y_tr, X_va_s, y_va, lr=0.05, epochs=60, batch_size=32)
t_ours = time.perf_counter() - t0
best = int(np.argmax([h["val_acc"] for h in hist_dg]))
print(f"mạng tự viết 64-128-64-10: {t_ours:.1f}s | val accuracy tốt nhất {hist_dg[best]['val_acc']:.4f} ở epoch {best + 1} | "
      f"test accuracy = {np.mean(net_dg.predict(X_te_s) == y_te):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot([h["loss"] for h in hist_dg], label="train")
axes[0].plot([h["val_loss"] for h in hist_dg], label="validation")
axes[0].set_title("cross-entropy")
axes[0].legend()
axes[1].plot([h["train_acc"] for h in hist_dg], label="train")
axes[1].plot([h["val_acc"] for h in hist_dg], label="validation")
axes[1].set_title("accuracy")
axes[1].legend()
plt.show()

In [ ]:
ref = {"MLPScratch (ml_course, Adam)": MLPScratch(hidden=(128, 64), lr=0.001, epochs=60, batch_size=32, seed=0),
       "sklearn MLPClassifier": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=0)}
for name, m in ref.items():
    m.fit(X_tr_s, y_tr)
    print(f"{name:30s} test accuracy = {np.mean(m.predict(X_te_s) == y_te):.4f}")

Mạng tự viết đạt độ chính xác ngang các bản cài đặt tham chiếu — không có "phép màu" nào trong thư viện, chỉ là các phép nhân ma trận và quy tắc chuỗi.

## 5. Overfitting khi ít dữ liệu

Chỉ dùng 150 ảnh để train, giữ nguyên mạng lớn:

In [ ]:
small = np.random.default_rng(0).choice(len(X_tr_s), 150, replace=False)
net_small = Network([64, 256, 256, 10], seed=0)
hist_small = train(net_small, X_tr_s[small], y_tr[small], X_va_s, y_va, lr=0.05, epochs=200, batch_size=32)
plt.figure(figsize=(8, 4))
plt.plot([h["loss"] for h in hist_small], label="train loss")
plt.plot([h["val_loss"] for h in hist_small], label="validation loss")
plt.legend()
plt.xlabel("epoch")
plt.show()
best_s = int(np.argmin([h["val_loss"] for h in hist_small]))
print(f"train accuracy cuối = {hist_small[-1]['train_acc']:.3f} | val accuracy cuối = {hist_small[-1]['val_acc']:.3f}")
print(f"val loss nhỏ nhất ở epoch {best_s + 1} ({hist_small[best_s]['val_loss']:.3f}); epoch cuối = {hist_small[-1]['val_loss']:.3f}")

Train accuracy 100%, loss train → 0, trong khi validation loss ngừng giảm từ khoảng giữa quá trình rồi nhích lên, val accuracy chỉ ~93% (so với ~95–97% khi train trên toàn bộ dữ liệu): khoảng cách lớn giữa train và validation — mạng **thuộc lòng** 150 ảnh. Chương 28 trình bày các công cụ chống lại: early stopping, weight decay, dropout, data augmentation, batch norm.

## 🧠 Kiểm tra nhanh

1. Mỗi lớp cần lưu gì trong `forward` để dùng ở `backward`?
2. Viết gradient của softmax + cross-entropy theo logit. Vì sao gộp hai bước?
3. Vì sao cần chuẩn hoá đầu vào trước khi huấn luyện?
4. Kiểm tra gradient thế nào? Lưu ý gì với ReLU?
5. Dấu hiệu nào trên đường cong loss cho thấy overfitting?

## 📌 Tóm tắt

- Mạng = chuỗi lớp; backprop = gọi `backward` ngược thứ tự, mỗi lớp nhận gradient đầu ra, trả gradient đầu vào.
- Softmax-CE gộp: gradient $\frac1n(\hat P - Y)$; luôn kiểm tra gradient.
- Độ rộng/sâu tăng khả năng biểu diễn; ít dữ liệu + mạng lớn → overfitting.

## 📚 Tài liệu tham khảo

- CS231n — *Neural Networks Part 1–3*, *Putting it together: Minimal Neural Network Case Study* (cs231n.github.io)
- Goodfellow, Bengio & Courville (2016) — *Deep Learning*, ch. 6.5 (back-propagation)
- Nielsen — *Neural Networks and Deep Learning*, ch. 1–2